# الجزء العاشر: المقابلات والمسار المهني

دفتر كود هذا الجزء — **3 فصلًا** و**14 كتلة بايثون**. الكتل مرتّبة كما وردت في الكتاب، ويمكن تشغيلها خليةً خلية.

**Colab:** File ▸ Upload notebook، ثم شغّل خلية الإعداد. **Kaggle:** أضف الدفتر ثم شغّل الخلايا.

**CPU/GPU:** كل شيء يعمل على CPU، لكن فصول التدريب أسرع على GPU (Colab: Runtime ▸ Change runtime type ▸ T4/A100 — Kaggle: Settings ▸ Accelerator ▸ GPU).

## فصول الجزء

- **الفصل 46:** أسئلة المقابلات — خريطة النجاة في بحر الأسئلة
- **الفصل 47:** أنماط تصميم أنظمة LLM — عندما يصبح النموذج مجرد لبنة
- **الفصل 48:** أفضل الممارسات الهندسية والمسار المهني — الرحلة لا تنتهي عند النشر

In [ ]:
# ⚙️ خلية الإعداد — شغّلها أولًا (Colab / Kaggle / محليًا)
# تثبّت ما ينقص فقط، وتتجاهل ما لا يدعم جهازك.
import sys, subprocess, importlib

PKGS = [
    ("torch", "torch"), ("transformers", "transformers"),
    ("datasets", "datasets"), ("accelerate", "accelerate"),
    ("peft", "peft"), ("trl", "trl"),
    ("sentence_transformers", "sentence-transformers"),
    ("sklearn", "scikit-learn"), ("pandas", "pandas"),
    ("numpy", "numpy"), ("matplotlib", "matplotlib"),
    ("tqdm", "tqdm"), ("evaluate", "evaluate"),
]
todo = []
for mod, pipname in PKGS:
    try:
        importlib.import_module(mod)
    except Exception:
        todo.append(pipname)
if todo:
    print("تثبيت الحزم الناقصة:", todo)
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *todo])
    except Exception as e:
        print("تعذّر تثبيت بعض الحزم (ثبّتها يدويًا إن احتجتها):", e)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"
print("device:", DEVICE)
print("جاهز ✅")


## الفصل 47: أنماط تصميم أنظمة LLM — عندما يصبح النموذج مجرد لبنة

الجزء 10 · عدد الكتل: 18

```
[المستخدم] → [Query] → [Retriever] → [Reranker] → [LLM] → [الإجابة]
                          ↑
                    [Vector DB]
                          ↑
                    [Indexing Pipeline]
                          ↑
                    [المستندات]
```

In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers import CrossEncoder
import numpy as np

class RAGSystem:
    def __init__(self, llm, embedder_name="intfloat/multilingual-e5-large",
                 reranker_name="BAAI/bge-reranker-v2-m3"):
        self.llm = llm
        self.embedder = SentenceTransformer(embedder_name)
        self.reranker = CrossEncoder(reranker_name)
        self.chunks = []
        self.embeddings = None

    def index(self, documents: list[str]):
        """فهرسة المستندات."""
        self.chunks = []
        for doc in documents:
            self.chunks.extend(self._chunk(doc))
        texts = [f"passage: {c}" for c in self.chunks]
        self.embeddings = self.embedder.encode(texts, normalize_embeddings=True)
        print(f"Indexed {len(self.chunks)} chunks")

    def _chunk(self, text: str, size: int = 500) -> list[str]:
        """تقسيم بسيط (استخدم تقسيمًا أفضل في الإنتاج)."""
        sentences = text.split(". ")
        chunks = []
        current = ""
        for sent in sentences:
            if len(current) + len(sent) <= size:
                current += sent + ". "
            else:
                chunks.append(current.strip())
                current = sent + ". "
        if current.strip():
            chunks.append(current.strip())
        return chunks

    def retrieve(self, query: str, k: int = 5) -> list[str]:
        """استرجاع + إعادة ترتيب."""
        # 1. استرجاع كثيف
        q_emb = self.embedder.encode([f"query: {query}"], normalize_embeddings=True)[0]
        scores = self.embeddings @ q_emb
        top_k = np.argsort(scores)[::-1][:k*2]
        candidates = [self.chunks[i] for i in top_k]

        # 2. إعادة ترتيب
        pairs = [(query, c) for c in candidates]
        rerank_scores = self.reranker.predict(pairs)
        ranked = sorted(zip(candidates, rerank_scores), key=lambda x: x[1], reverse=True)
        return [c for c, _ in ranked[:k]]

    def generate(self, query: str) -> str:
        """توليد الإجابة."""
        contexts = self.retrieve(query)
        context_text = "\n\n".join([f"[مصدر {i+1}]: {c}" for i, c in enumerate(contexts)])

        prompt = f"""أجب على السؤال بناءً على السياق فقط.
إذا لم تجد الإجابة، قل "لا أعرف".

السياق:
{context_text}

السؤال: {query}

الإجابة:"""
        return self.llm(prompt)


```
[المستخدم] → [Agent Loop] → [Tool Calling] → [Environment]
                  ↑               ↓
                  └── [Observation] ←─┘
```

In [ ]:
import json
import re

class ReActAgent:
    def __init__(self, llm, tools: dict, max_steps: int = 5):
        self.llm = llm
        self.tools = tools
        self.max_steps = max_steps

    def run(self, query: str) -> str:
        history = []
        for step in range(self.max_steps):
            prompt = self._build_prompt(query, history)
            response = self.llm(prompt)

            # هل انتهى؟
            if "Final Answer:" in response:
                return response.split("Final Answer:")[-1].strip()

            # استخرج الأداة
            action, action_input = self._parse_action(response)
            if not action:
                history.append("Observation: لم أفهم. حاول مرة أخرى.")
                continue

            # نفذ
            if action not in self.tools:
                observation = f"الأداة {action} غير موجودة."
            else:
                try:
                    observation = str(self.tools[action](action_input))
                except Exception as e:
                    observation = f"خطأ: {e}"

            history.append(response.strip())
            history.append(f"Observation: {observation}")

        return "لم أتمكن من الإجابة."

    def _build_prompt(self, query, history):
        tools_desc = "\n".join([f"- {name}: {func.__doc__}" for name, func in self.tools.items()])
        history_text = "\n".join(history) if history else "(لا يوجد)"
        return f"""أنت وكيل. استخدم الأدوات للإجابة.

الأدوات:
{tools_desc}

التنسيق:
Thought: [تفكيرك]
Action: [اسم الأداة]
Action Input: [المدخل]

عندما تكون جاهزًا:
Final Answer: [الإجابة]

السؤال: {query}

التاريخ:
{history_text}
"""

    def _parse_action(self, response):
        action_match = re.search(r"Action:\s*(\w+)", response)
        input_match = re.search(r"Action Input:\s*(.+)", response, re.DOTALL)
        if not action_match:
            return None, None
        return action_match.group(1), input_match.group(1).strip() if input_match else ""


```
[Manager] → [Researcher] → [Writer] → [Reviewer]
    ↑                                       ↓
    └───────────────────────────────────────┘
```

In [ ]:
class MultiAgentSystem:
    def __init__(self, manager_llm, workers: dict):
        self.manager = manager_llm
        self.workers = workers

    def run(self, task: str) -> str:
        prompt = f"""أنت مدير. لديك الفريق:
{list(self.workers.keys())}

المهمة: {task}

اختر الوكيل المناسب، وأعد اسمه فقط."""
        worker_name = self.manager(prompt).strip()

        if worker_name not in self.workers:
            return f"لا يوجد وكيل باسم {worker_name}"

        return self.workers[worker_name].run(task)


```
[جمع البيانات] → [تنظيف] → [تنسيق SFT] → [تدريب LoRA] → [تقييم] → [نشر]
```

In [ ]:
def fine_tuning_pipeline(
    data_path: str,
    base_model: str,
    output_dir: str,
    eval_fn,
    min_accuracy: float = 0.8,
):
    """
    خط أنابيب: تحميل، تدريب، تقييم، نشر.
    """
    import pandas as pd
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, get_peft_model
    from trl import SFTTrainer, SFTConfig
    from datasets import Dataset

    # 1. تحميل
    df = pd.read_csv(data_path)

    # 2. تنسيق
    ds = Dataset.from_pandas(df)

    # 3. تدريب
    tokenizer = AutoTokenizer.from_pretrained(base_model)
    model = AutoModelForCausalLM.from_pretrained(base_model, torch_dtype="bfloat16", device_map="auto")
    lora = LoraConfig(r=16, lora_alpha=16, target_modules="all-linear", task_type="CAUSAL_LM")
    model = get_peft_model(model, lora)

    trainer = SFTTrainer(
        model=model,
        args=SFTConfig(output_dir=output_dir, num_train_epochs=2, bf16=True),
        train_dataset=ds,
        tokenizer=tokenizer,
    )
    trainer.train()

    # 4. تقييم
    accuracy = eval_fn(model, tokenizer)
    if accuracy < min_accuracy:
        raise Exception(f"Accuracy {accuracy:.2f} < {min_accuracy}. Blocking deployment.")

    # 5. حفظ
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    return model, tokenizer


```
[Offline Eval] → [Online Eval] → [A/B Testing] → [Feedback Loop]
```

In [ ]:
class EvalPipeline:
    def __init__(self, judge_llm, test_cases: list[dict]):
        self.judge = judge_llm
        self.test_cases = test_cases

    def run_offline(self, model, tokenizer) -> dict:
        """تقييم آلي على مجموعة اختبار."""
        scores = []
        for case in self.test_cases:
            # توليد
            answer = generate(model, tokenizer, case["input"])
            # تقييم
            score = self._judge(case["input"], answer, case.get("expected"))
            scores.append(score)
        return {
            "mean_score": sum(scores) / len(scores),
            "min_score": min(scores),
            "max_score": max(scores),
        }

    def _judge(self, question, answer, expected):
        """LLM-as-a-Judge."""
        prompt = f"""قيّم الإجابة على مقياس 1-5.

السؤال: {question}
الإجابة: {answer}
الإجابة المتوقعة (إن وجدت): {expected}

أعد رقمًا فقط."""
        result = self.judge(prompt).strip()
        try:
            return int(result)
        except:
            return 0

    def run_online(self, user_feedback: list[dict]) -> dict:
        """تقييم من ملاحظات المستخدمين."""
        positive = sum(1 for f in user_feedback if f["rating"] >= 4)
        return {
            "satisfaction": positive / len(user_feedback) if user_feedback else 0,
            "total": len(user_feedback),
        }


```
[Load Balancer] → [vLLM Pod 1] → [GPU]
                → [vLLM Pod 2] → [GPU]
                → [vLLM Pod 3] → [GPU]
```

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
from openai import OpenAI

app = FastAPI()
client = OpenAI(base_url="http://vllm:8000/v1", api_key="dummy")

class Request(BaseModel):
    prompt: str
    max_tokens: int = 100

@app.post("/generate")
def generate(req: Request):
    response = client.chat.completions.create(
        model="Qwen/Qwen2.5-7B-Instruct",
        messages=[{"role": "user", "content": req.prompt}],
        max_tokens=req.max_tokens,
    )
    return {"text": response.choices[0].message.content}

@app.get("/health")
def health():
    return {"status": "ok"}


```
[Input] → [Input Filter] → [LLM] → [Output Filter] → [Output]
                ↓                            ↓
          [Blocked]                    [Redacted]
```

In [ ]:
import re

class Guardrails:
    def __init__(self):
        self.injection_patterns = [
            r"تجاهل\s+(كل\s+)?التعليمات",
            r"ignore\s+(all\s+)?instructions",
            r"system\s*:",
        ]
        self.pii_patterns = {
            "email": r"\b[\w.-]+@[\w.-]+\.\w+\b",
            "phone": r"\b(?:\+966|0)?5\d{8}\b",
        }

    def check_input(self, text: str) -> tuple[bool, str]:
        """يفلتر المدخلات."""
        for pattern in self.injection_patterns:
            if re.search(pattern, text, re.IGNORECASE):
                return False, "injection_detected"
        return True, ""

    def check_output(self, text: str) -> tuple[bool, str]:
        """يفلتر المخرجات."""
        for name, pattern in self.pii_patterns.items():
            if re.search(pattern, text):
                text = re.sub(pattern, f"[{name.upper()}_REDACTED]", text)
        return True, text


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

class SemanticCache:
    def __init__(self, embedder, threshold=0.95):
        self.embedder = embedder
        self.threshold = threshold
        self.cache = []  # list of (embedding, response)

    def get(self, query: str):
        if not self.cache:
            return None
        q_emb = self.embedder.encode([f"query: {query}"], normalize_embeddings=True)[0]
        for emb, response in self.cache:
            sim = np.dot(q_emb, emb)
            if sim >= self.threshold:
                return response
        return None

    def set(self, query: str, response: str):
        q_emb = self.embedder.encode([f"query: {query}"], normalize_embeddings=True)[0]
        self.cache.append((q_emb, response))


```
[Primary Model] → [Fail] → [Fallback Model] → [Response]
       ↓
   [Success]
```

In [ ]:
def generate_with_fallback(prompt, primary, fallback, timeout=5):
    try:
        return primary.generate(prompt, timeout=timeout)
    except Exception as e:
        print(f"Primary failed: {e}. Using fallback.")
        return fallback.generate(prompt)


```
[المستخدم]
    ↓
[Guardrails] → [Cache] → [RAG] → [Agent] → [LLM] → [Guardrails] → [المستخدم]
    ↑                                                      ↓
[Monitoring] ←────────────── [Evaluation] ←──────────────┘
```

## الفصل 48: أفضل الممارسات الهندسية والمسار المهني — الرحلة لا تنتهي عند النشر

الجزء 10 · عدد الكتل: 9

In [ ]:
import mlflow
import mlflow.pytorch

def train_and_log(model, train_loader, config):
    """يدرب ويسجل كل شيء في MLflow."""
    mlflow.set_experiment("sentiment-arabic")

    with mlflow.start_run(run_name=f"run-{config['version']}"):
        # سجل الإعدادات
        mlflow.log_params(config)

        # سجل الإصدارات
        mlflow.set_tags({
            "model_version": config["version"],
            "dataset_version": config["dataset_version"],
            "prompt_version": config.get("prompt_version", "v1"),
            "git_commit": get_git_commit(),
        })

        # تدريب
        for epoch in range(config["epochs"]):
            loss = train_epoch(model, train_loader)
            mlflow.log_metric("train_loss", loss, step=epoch)

        # سجل النموذج
        mlflow.pytorch.log_model(model, "model")

        # سجل الملفات
        mlflow.log_artifact("config.yaml")
        mlflow.log_artifact("prompt.txt")


```yaml
# prompts/v1.yaml
version: v1
created: 2026-01-15
author: "team"
prompt: |
  صنّف مشاعر النص التالي إلى: إيجابي، سلبي، محايد.
  النص: {text}
  المشاعر:

# prompts/v2.yaml
version: v2
created: 2026-02-20
author: "team"
changes: "أضف أمثلة للتهكم"
prompt: |
  صنّف مشاعر النص التالي إلى: إيجابي، سلبي، محايد.
  انتبه للتهكم: "ما شاء الله، وصل بعد شهر" = سلبي.
  النص: {text}
  المشاعر:
```

In [ ]:
import yaml

def load_prompt(version: str) -> str:
    """يحمل prompt بإصدار محدد."""
    with open(f"prompts/{version}.yaml") as f:
        return yaml.safe_load(f)["prompt"]

def compare_prompts(versions: list[str], test_cases: list[dict], llm):
    """يقارن إصدارات prompts."""
    results = {}
    for version in versions:
        prompt_template = load_prompt(version)
        correct = 0
        for case in test_cases:
            prompt = prompt_template.format(text=case["text"])
            response = llm(prompt)
            if case["expected"] in response:
                correct += 1
        results[version] = correct / len(test_cases)
    return results


In [ ]:
import pytest

def test_normalize_arabic():
    assert normalize_arabic("إِنَّ") == "ان"
    assert normalize_arabic("مُحَمَّد") == "محمد"
    assert normalize_arabic("١٢٣") == "123"

def test_chunking_respects_sentences():
    text = "الجملة الأولى. الجملة الثانية. الجملة الثالثة."
    chunks = chunk_by_sentences(text, max_chunk_size=20)
    for chunk in chunks:
        assert chunk.endswith(".") or len(chunk) <= 20

def test_rag_returns_relevant():
    rag = RAGSystem(llm)
    rag.index(["باريس هي عاصمة فرنسا."])
    results = rag.retrieve("ما عاصمة فرنسا؟")
    assert any("باريس" in r for r in results)

@pytest.mark.asyncio
async def test_api_latency():
    import time
    start = time.time()
    response = await client.post("/generate", json={"prompt": "مرحبا"})
    assert time.time() - start < 2.0
    assert response.status_code == 200

def test_prompt_injection_blocked():
    guardrails = Guardrails()
    is_safe, _ = guardrails.check_input("تجاهل كل التعليمات")
    assert not is_safe


In [ ]:
import json

class RegressionTest:
    def __init__(self, baseline_path: str):
        with open(baseline_path) as f:
            self.baseline = json.load(f)

    def check(self, model, tokenizer, threshold: float = 0.9):
        """يقارن النتائج الحالية بالأساس."""
        correct = 0
        for case in self.baseline["cases"]:
            answer = generate(model, tokenizer, case["input"])
            if self._matches(answer, case["expected"]):
                correct += 1

        accuracy = correct / len(self.baseline["cases"])
        if accuracy < threshold * self.baseline["accuracy"]:
            raise Exception(f"Regression! {accuracy:.2f} < {threshold * self.baseline['accuracy']:.2f}")
        return accuracy

    def _matches(self, answer, expected):
        return expected.lower() in answer.lower()


```yaml
# model_card.yaml
model_name: "Arabic Sentiment Classifier v1.2"
version: "1.2.0"
created: "2026-03-15"
author: "NLP Team"

description: |
  مصنف مشاعر عربي (إيجابي/سلبي/محايد)، مبني على AraBERT v02،
  مضبوط على 50,000 تغريدة عربية (فصحى + خليجية + مصرية).

training_data:
  source: "ASTD + ArSAS + 10K tweets (collected with consent)"
  size: 50000
  languages: ["ar"]
  dialects: ["MSA", "Gulf", "Egyptian"]

performance:
  test_f1_macro: 0.84
  test_accuracy: 0.86
  per_class:
    positive: {f1: 0.88, precision: 0.87, recall: 0.89}
    negative: {f1: 0.82, precision: 0.84, recall: 0.80}
    neutral: {f1: 0.81, precision: 0.80, recall: 0.82}

limitations:
  - "ضعيف في التهكم (F1 ~0.65)"
  - "لم يُختبر على اللهجة المغربية"
  - "قد يخطئ في النصوص القصيرة جدًا (< 3 كلمات)"

bias_analysis:
  gender: "لا يظهر تحيز جنساني كبير في الاختبار"
  dialect: "أداء أفضل على الخليجية من المصرية (-5% F1)"

intended_use:
  - "تحليل مشاعر مراجعات المنتجات"
  - "مراقبة رضا العملاء"

not_intended_for:
  - "قرارات طبية أو قانونية"
  - "تحليل مشاعر فردية (قد يخطئ)"

contact: "nlp-team@company.com"
license: "Internal use only"
```

```markdown
# Decision Log

## 2026-02-10: اختيار AraBERT بدلًا من Qwen2.5-7B

**السياق**: نحتاج إلى مصنف مشاعر، والحجم مهم (زمن استجابة < 100ms).

**الخيارات**:
- AraBERT (110M): F1 0.86، زمن 30ms، تكلفة منخفضة.
- Qwen2.5-7B (LoRA): F1 0.88، زمن 300ms، تكلفة 10×.

**القرار**: AraBERT. الفرق في F1 (2 نقاط) لا يبرر التكلفة والزمن.

**المراجعة**: بعد 3 أشهر، إذا احتجنا إلى دقة أعلى، سنعيد التقييم.

**المسؤول**: فريق NLP
```

In [ ]:
# مستودع شخصي لتجارب LLM
# experiments/
# ├── 2026-01-15-lora-vs-qlora/
# │   ├── README.md (السؤال، الطريقة، النتائج)
# │   ├── config.yaml
# │   ├── results.json
# │   └── notes.md
# ├── 2026-02-20-rag-chunking/
# └── ...

def log_experiment(name: str, config: dict, results: dict, notes: str):
    """يسجل تجربة في المستودع الشخصي."""
    import json, os, datetime
    date = datetime.datetime.now().strftime("%Y-%m-%d")
    path = f"experiments/{date}-{name}"
    os.makedirs(path, exist_ok=True)

    with open(f"{path}/config.yaml", "w") as f:
        import yaml
        yaml.dump(config, f)

    with open(f"{path}/results.json", "w") as f:
        json.dump(results, f, indent=2)

    with open(f"{path}/notes.md", "w") as f:
        f.write(notes)

    print(f"Logged: {path}")


```
        ← عرض (معرفة عامة) →
              ┌─────────────┐
              │  فهم الأعمال │
              │  التواصل     │
              │  الهندسة     │
              │  البيانات    │
              └─────────────┘
                    │
                    │ عمق
                    │ (تخصص)
                    ▼
              LLM / NLP
```